# 🎓 Autonomous Scholarship Agent (With Deep Research Planner)

## 1. Imports & Setup

In [ ]:
import os
import asyncio
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail
from agents.model_settings import ModelSettings
from agents import function_tool
from agents import Agent, Runner, trace, input_guardrail, GuardrailFunctionOutput, WebSearchTool

print("✅ All libraries imported.")

## 2. Environment Setup

In [ ]:
load_dotenv()

config = {
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    "SENDGRID_API_KEY": os.getenv("SENDGRID_API_KEY"),
    "SENDER_EMAIL": os.getenv("SENDER_EMAIL"),
}

sg = SendGridAPIClient(config['SENDGRID_API_KEY'])

print("✅ Environment loaded and SendGrid client initialized.")

## 3. Structured Output Models (Updated with Planner)

In [ ]:
class ApplicantProfile(BaseModel):
    full_name: str
    university: str
    program: str
    research_interest: str
    academic_background: str
    work_experience: str

class ScholarshipGuardrailOutput(BaseModel):
    is_safe: bool = Field(..., description="True if the application is safe, False otherwise.")
    violations: List[str] = Field(default_factory=list, description="List of detected violations, if any.")


class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

print("✅ Pydantic models defined (including Search Plan).")

## 4. Deep Research Agents (Planner & Executor)

In [ ]:
HOW_MANY_SEARCHES = 3
INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

# 2. The Searcher: Actually does the Googling
search_agent = Agent(
    name="SearchAgent",
    instructions="You are a web searcher. Execute the query provided and return the raw results.",
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required")
)

# --- Research Hlper Fx ---

async def perform_deep_research(university: str, program: str, interest: str):
    """ 
    1. Plan the searches
    2. Execute them in parallel
    3. Return the combined findings
    """
    query = f"Find key details about the {program} at {university} specifically for someone interested in {interest}. Look for professor names, lab names, and mission statements."
    
    # A. PLAN
    print(f"🤔 Planning research for: {university}...")
    plan_result = await Runner.run(planner_agent, query)
    search_plan = plan_result.final_output
    
    # B. EXECUTE (Loop through the plan)
    results = []
    print(f"🌐 Executing {len(search_plan.searches)} searches in parallel...")
    
    # Create a list of async tasks
    tasks = []
    for item in search_plan.searches:
        print(f"   -> Searching: {item.query}")
        tasks.append(Runner.run(search_agent, item.query))
    
    # Run them all at once
    search_outputs = await asyncio.gather(*tasks)
    
    # C. COMBINE
    final_context = "\n\n".join([res.final_output for res in search_outputs])
    return final_context

print("✅ Planner and Search agents defined.")

## 5. Application Writer Agents

In [ ]:
formal_agent = Agent(
    name="Formal Academic Writer",
    instructions="""
You are an expert academic writer specializing in formal, rigorous scholarship applications. Your style is formal, research-focused, and emphasizes academic achievement and intellectual contribution.
""",
    model="gpt-4o"
)

motivational_agent = Agent(
    name="Motivational Personal Writer",
    instructions="""
You are an expert narrative writer specializing in emotionally compelling, motivational scholarship applications. Your style is warm, authentic, and personally engaging.
""",
    model="gpt-4o"
)

concise_agent = Agent(
    name="Concise Professional Writer",
    instructions="""
You are an expert professional communicator specializing in concise, results-oriented scholarship applications. Your style is direct, clear, and action-oriented.
""",
    model="gpt-4o"
)

print("✅ Writer agents defined")

## 6. Agent-as-Tools Conversion

In [ ]:
formal_writer_tool = formal_agent.as_tool()
motivational_writer_tool = motivational_agent.as_tool()
concise_writer_tool = concise_agent.as_tool()

print("✅ Writer agents converted to tools")

## 7. Input Guardrail Agent

In [ ]:
guardrail_agent = Agent(
    name="Scholarship Application Guardrail",
    instructions="""You are a guardrail agent. Your job is to check a scholarship application for forbidden content like fake professor names (e.g., 'Dr. AI', 'Prof. ChatGPT'), explicit impersonation, or disallowed claims (e.g., 'I am already admitted'). If any violations are found, set is_safe to False and list the violations.""",
    output_type=ScholarshipGuardrailOutput,
    model="gpt-4o-mini"
)

@input_guardrail
async def check_scholarship_application_safety(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_safe = result.final_output.is_safe
    return GuardrailFunctionOutput(
        output_info={"violations": result.final_output.violations},
        tripwire_triggered=not is_safe
    )

print("✅ Agent-based guardrail defined")

## 8. Email Manager Agent & Tools

In [ ]:
@function_tool
def send_email(recipient: str, subject: str, html_body: str) -> dict:
    """
    Sends an email using SendGrid.
    Args:
        recipient: The email address of the receiver.
        subject: The subject line of the email.
        html_body: The content of the email in HTML format.
    """
    
    message = Mail(
        from_email=config['SENDER_EMAIL'],
        to_emails=recipient,
        subject=subject,
        html_content=html_body
    )
    try:
        response = sg.send(message)
        return {'status': 'success', 'id': response.headers.get('X-Message-Id', None)}
    except Exception as e:
        return {'status': 'error', 'error': str(e)}

email_manager = Agent(
    name="Email Manager",
    instructions="""
    You are the Email Manager. You receive a scholarship application body.
    1. Write a professional, engaging subject line for this application.
    2. Convert the body to clean HTML.
    3. Use the send_email tool to send it to the applicant's email.
    """,
    tools=[send_email],
    model="gpt-4o"
)

print("✅ Email Manager agent defined (Correctly using @function_tool)")

## 9. Scholarship Manager Agent (Coordinator)

In [ ]:
scholarship_manager = Agent(
    name="Scholarship Manager",
    instructions="""
    You are the Scholarship Manager. 
    1. You are given an applicant's profile AND detailed research notes about the university.
    2. Call all three writer tools (Formal, Motivational, Concise) to generate drafts.
    3. Compare their outputs. Ensure the writers used the RESEARCH NOTES (e.g. professor names) in their drafts.
    4. Select the SINGLE best draft.
    5. Handoff that winning draft to the 'Email Manager' agent to handle the sending.
    Do not output the final text yourself; let the Email Manager handle it.
    """,
    tools=[formal_writer_tool, motivational_writer_tool, concise_writer_tool],
    handoffs=[email_manager], 
    input_guardrails=[check_scholarship_application_safety],
    model="gpt-4o"
)

print("✅ Scholarship Manager agent defined with guardrails and handoff")

## 10. End-to-End Orchestration (Plan -> Search -> Draft -> Send)

In [ ]:
async def process_scholarship_application(applicant_data: dict, recipient_email: str):
    """
    Orchestrate the full scholarship application process.
    """
    
    # STEP 1: DEEP RESEARCH (PLAN & EXECUTE)
    uni_briefing = await perform_deep_research(
        applicant_data['university'], 
        applicant_data['program'],
        applicant_data['research_interest']
    )
    print(f"✅ Research Complete. Context length: {len(uni_briefing)} chars.")

    # STEP 2: COMPOSE PROMPT
    prompt = f"""
    Compose a scholarship application for:
    Name: {applicant_data['full_name']}
    University: {applicant_data['university']}
    Program: {applicant_data['program']}
    
    CRITICAL CONTEXT FROM RESEARCH (Use these facts to make the essay specific):
    {uni_briefing}
    
    My Profile:
    Research Interest: {applicant_data['research_interest']}
    Academic Background: {applicant_data['academic_background']}
    Work Experience: {applicant_data['work_experience']}

    Send the final email to: {recipient_email}
    """
    
    # STEP 3: RUN THE MANAGER
    result = await Runner.run(scholarship_manager, prompt)
    return result

print("✅ End-to-end application orchestration complete")

## 11. Run Test Cases

In [ ]:
async def run_test():
    recipient = "samandari@gmail.com"

    # Allowed case
    allowed_applicant = {
        "full_name": "Sarah Chen",
        "university": "MIT",
        "program": "Master's in Artificial Intelligence",
        "research_interest": "Natural language processing and multilingual AI systems",
        "academic_background": "BS Computer Science from University of California, GPA 3.9",
        "work_experience": "2 years at Google Brain as ML Engineer"
    }
    print("\n--- TESTING ALLOWED CASE ---")
    allowed_result = await process_scholarship_application(allowed_applicant, recipient)
    print(allowed_result)

In [ ]:
# Use await in Jupyter instead of asyncio.run()
await run_test()